# LmrR EVB Parametrization — Fitting H12 and Delta from QM RS/TS/PS Data

This notebook takes the QM outputs you already have (RS/TS/PS energies, optimized
geometries, and partial charges from the xTB/Gaussian workflow) and uses them to
fit the two core EVB parameters for each reaction step:

- **Delta** — the constant energy shift applied to the product-like diabatic
  surface (state 2), fit from the QM reaction energy (RS -> PS).
- **H12** — the off-diagonal coupling between the two diabatic (valence-bond)
  states, fit from the QM barrier at the TS.

## Method (short version)

Your QM energies (RS/TS/PS) are **adiabatic** — already-mixed ground-state
energies. EVB needs two **diabatic** surfaces:

- State 1 = reactant-like bonding (the bonds present in RS)
- State 2 = product-like bonding (the bonds present in PS)

evaluated at the *same* geometry. We build a reduced diabatic energy for each
state using Morse potentials only on the bonds that are actually changing
(`bond_changes`, per step), since those are the only bonds whose character
differs between state 1 and state 2. Bonds tagged `weakened`/`strengthened`
are included in **both** states (present in both topologies, just with a
different equilibrium length in each), while `formed`/`broken`/`double-bond
formed` bonds are state-specific.

Given the diabatic energies e1(R) and e2(R), the ground state of the 2x2 EVB
Hamiltonian is:

```
Eg(R) = 0.5*(e1+e2) - sqrt( (0.5*(e1-e2))**2 + H12**2 )
```

We solve:
1. **Delta** from the PS geometry, assuming state 2 dominates there
   (`Eg(PS) ~= e2(PS) + Delta = QM reaction energy`).
2. **H12** from the TS geometry, algebraically inverting the ground-state
   formula against the QM barrier.

## Caveats (read before trusting the numbers)

- The Morse parameters (bond dissociation energy `De`, force constant `k`)
  come from a small literature/typical-value lookup table by element pair,
  **not** from your actual Hessians. Treat this as a first-pass / illustrative
  EVB parametrization, not a production-ready one.
- Only the reactive bonds are modeled — solvent/protein electrostatics and the
  rest of the force field are assumed to (approximately) cancel between the
  two states. Your RS/PS partial charges are carried through for reference and
  for the next step (building the full EVB/MM force field), but are not yet
  folded into the H12/Delta algebra here.
- This uses 3 QM points per step (RS, TS, PS). If you have a relaxed scan or
  extra constrained-TS geometries (see the constrained `optimize_structure()`
  fix), you can extend `fit_delta_and_h12` to fit against more points instead
  of solving the 2-point/3-point system exactly.
- If you have higher-level (DFT/RESP) single-point energies from your
  `SP_RESP_Barriers` workflow, use those for `E_RS/E_TS/E_PS` instead of raw
  xTB energies — H12 will only be as good as the barrier you feed it.


## 1. Configuration

Edit `BASE_DIR` / `STAGE2_DIR` to match your machine. This expects the same directory layout as `LmrR_5QM_xTB_Gaussian_WORKFLOW_FIXED.ipynb` Stage 2 (`{step}_{role}_xtbopt.xyz`, `{step}_{role}_charges.csv`, and the `stage2_75_calculations.csv` master results table).

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Paths — match the Stage 2 layout from LmrR_5QM_xTB_Gaussian_WORKFLOW_FIXED.ipynb
# ------------------------------------------------------------------

BASE_DIR = Path(r"D:\PhD_Thesis\LmrR_EVB")
CHARGES_DIR = BASE_DIR / "charges"

XTB_DIR = CHARGES_DIR / "fast_lmrr_5QM" / "xtb_preparation"
STAGE2_DIR = XTB_DIR / "stage2_five_xtb_protocols"

# Which Stage-2 protocol's geometries/energies/charges to use for the fit.
# Must be one of the PROTOCOLS keys from the Stage 2 notebook.
PROTOCOL = "GFN2_tight"

HARTREE_TO_KCAL = 627.5094740631

ROLES = ["RS", "TS", "PS"]


## 2. Reaction steps and bond changes

Self-contained copy of the `bond_changes` you already defined per step, so this notebook doesn't depend on run order in the other notebooks.

In [2]:
@dataclass(frozen=True)
class BondChange:
    kind: str        # "formed" | "broken" | "double-bond formed" | "weakened" | "strengthened"
    serial_i: int
    serial_j: int
    note: str


@dataclass(frozen=True)
class StructureSet:
    label: str
    charge: int
    multiplicity: int
    bond_changes: tuple[BondChange, ...]
    comment: str


STRUCTURE_SETS: dict[str, StructureSet] = {
    "RS1_1_to_TS1_2": StructureSet(
        label="RS1_1_to_TS1_2",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("formed", 1, 10, "PAF.N2 attacks ENL.C10; N-C bond formation"),
            BondChange("weakened", 10, 16, "ENL.C10=O1 carbonyl weakens"),
        ),
        comment="Step 1.1 addition. TS is a starting TS guess; PS is the tetrahedral state.",
    ),
    "TS1_2_to_PS1_2b": StructureSet(
        label="TS1_2_to_PS1_2b",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 1, 29, "PAF N-H breaks"),
            BondChange("formed", 27, 29, "W1 accepts H29"),
            BondChange("broken", 60, 61, "W2 O-H breaks"),
            BondChange("formed", 16, 61, "ENL.O1 accepts H61"),
        ),
        comment="Two-water proton redistribution.",
    ),
    "RS1_2b_to_PS1_3": StructureSet(
        label="RS1_2b_to_PS1_3",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 27, 29, "W1 hydronium O-H breaks"),
            BondChange("formed", 16, 29, "carbinolamine O accepts H29"),
            BondChange("broken", 10, 16, "C-O leaving-water bond breaks"),
            BondChange("double-bond formed", 1, 10, "iminium N=C forms"),
        ),
        comment="Dehydration to iminium.",
    ),
    "RS2_1_to_TS2_1a": StructureSet(
        label="RS2_1_to_TS2_1a",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("formed", 12, 20, "new C-C sigma bond ENL.C12--IND.C3"),
            BondChange("weakened", 1, 10, "iminium N=C weakens"),
            BondChange("strengthened", 10, 11, "C10-C11 bond strengthens"),
        ),
        comment="Friedel-Crafts C-C bond formation.",
    ),
    "TS2_1a_to_PS2_2": StructureSet(
        label="TS2_1a_to_PS2_2",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 20, 50, "IND.C3-H50 breaks"),
            BondChange("formed", 60, 50, "W2 accepts H50"),
            BondChange("broken", 27, 58, "W1 O-H58 breaks"),
            BondChange("formed", 11, 58, "ENL.C11 accepts H58"),
            BondChange("double-bond formed", 1, 10, "N=C/enamine bond order changes"),
        ),
        comment="Combined 2.2 tautomerization (2.2a RS -> 2.2b TS -> 2.2b PS).",
    ),
}

STEPS = list(STRUCTURE_SETS)

STATE1_KINDS = {"broken", "weakened", "strengthened"}   # bonds present in RS
STATE2_KINDS = {"formed", "double-bond formed", "weakened", "strengthened"}  # bonds present in PS

print("Steps:", STEPS)


Steps: ['RS1_1_to_TS1_2', 'TS1_2_to_PS1_2b', 'RS1_2b_to_PS1_3', 'RS2_1_to_TS2_1a', 'TS2_1a_to_PS2_2']


## 3. Geometry / charge / energy loaders

Assumes PDB serial == XYZ line index, matching the Stage 2 notebook convention. If your PDBs have gaps, swap in the real `serial_to_xyz` map from the `*_pdb_to_xyz_map.csv` files instead.

In [3]:
def read_xyz(path: Path):
    lines = path.read_text(encoding="utf-8").splitlines()
    n = int(lines[0].strip())
    labels, coords = [], []
    for line in lines[2 : 2 + n]:
        parts = line.split()
        labels.append(parts[0])
        coords.append((float(parts[1]), float(parts[2]), float(parts[3])))
    return labels, coords


def distance(coords, i_serial, j_serial):
    xi, yi, zi = coords[i_serial - 1]
    xj, yj, zj = coords[j_serial - 1]
    return math.sqrt((xi - xj) ** 2 + (yi - yj) ** 2 + (zi - zj) ** 2)


def geometry_path(step: str, role: str) -> Path:
    return STAGE2_DIR / PROTOCOL / step / f"{step}_{role}_xtbopt.xyz"


def charges_path(step: str, role: str) -> Path:
    return STAGE2_DIR / PROTOCOL / step / f"{step}_{role}_charges.csv"


def load_charges(step: str, role: str) -> pd.DataFrame | None:
    path = charges_path(step, role)
    if not path.exists():
        return None
    return pd.read_csv(path)


def load_qm_energies_from_master(master_csv: Path, protocol: str) -> pd.DataFrame:
    '''Read the Stage 2 `stage2_75_calculations.csv` master table and pivot
    to one row per step with RS/TS/PS energies (Hartree) for `protocol`.
    '''
    df = pd.read_csv(master_csv)
    df = df[(df["protocol"] == protocol) & (df["status"] == "SUCCESS")]
    out = {}
    for step in STEPS:
        sub = df[df["step"] == step]
        row = {"step": step}
        for role in ROLES:
            match = sub[sub["role"] == role]
            row[f"E_{role}_Eh"] = match["energy_Eh"].iloc[0] if len(match) else np.nan
        out[step] = row
    return pd.DataFrame(out.values())


MASTER_CSV = STAGE2_DIR / "stage2_75_calculations.csv"

if MASTER_CSV.exists():
    qm_energies_df = load_qm_energies_from_master(MASTER_CSV, PROTOCOL)
else:
    print(f"Master results file not found:\n{MASTER_CSV}")
    print("Falling back to an empty table — fill in qm_energies_df manually, e.g.:")
    print('  qm_energies_df = pd.DataFrame([{"step": "RS1_1_to_TS1_2", '
          '"E_RS_Eh": -123.456, "E_TS_Eh": -123.400, "E_PS_Eh": -123.480}, ...])')
    qm_energies_df = pd.DataFrame(columns=["step", "E_RS_Eh", "E_TS_Eh", "E_PS_Eh"])

qm_energies_df


,step,E_RS_Eh,E_TS_Eh,E_PS_Eh
0,RS1_1_to_TS1_2,-83.440790,-83.438995,-83.437279
1,TS1_2_to_PS1_2b,-83.439228,-83.439179,NaN
2,RS1_2b_to_PS1_3,NaN,NaN,NaN
3,RS2_1_to_TS2_1a,NaN,NaN,NaN
4,TS2_1a_to_PS2_2,-110.267327,-110.273360,-110.295016


## 4. Morse diabatic bond parameters

Approximate literature bond-dissociation energies (`De`, kcal/mol) and stretching force constants (`k`, kcal/mol/A^2), keyed by element pair. Override `BOND_PARAMS` with better values (e.g. from your own Hessians) whenever you have them — everything downstream just reads this table.

In [4]:
# (element_a, element_b) -> (k [kcal/mol/A^2], De [kcal/mol])
# Sorted element pairs; typical single/double-bond values from common
# organic/biomolecular force fields and experimental BDEs.
BOND_PARAMS: dict[tuple[str, str], tuple[float, float]] = {
    ("C", "C"): (310.0, 83.0),
    ("C", "N"): (337.0, 73.0),
    ("C", "O"): (320.0, 85.0),
    ("O", "H"): (553.0, 111.0),
    ("N", "H"): (434.0, 93.0),
    ("C", "H"): (340.0, 99.0),
    ("O", "O"): (300.0, 35.0),
}

# Extra entries used when a bond_change is tagged "double-bond formed":
# treat as stiffer/stronger than the corresponding single bond.
DOUBLE_BOND_PARAMS: dict[tuple[str, str], tuple[float, float]] = {
    ("C", "N"): (615.0, 147.0),
    ("C", "C"): (549.0, 146.0),
    ("C", "O"): (570.0, 173.0),
}

DEFAULT_PARAMS = (400.0, 80.0)  # fallback (k, De) if a pair isn't in the table


def element_pair(labels, serial_i, serial_j):
    ei = labels[serial_i - 1].capitalize()
    ej = labels[serial_j - 1].capitalize()
    return tuple(sorted((ei, ej)))


def morse_params(labels, bc: BondChange):
    pair = element_pair(labels, bc.serial_i, bc.serial_j)
    table = DOUBLE_BOND_PARAMS if bc.kind == "double-bond formed" else BOND_PARAMS
    k, De = table.get(pair, BOND_PARAMS.get(pair, DEFAULT_PARAMS))
    if pair not in table and pair not in BOND_PARAMS:
        print(f"    (no bond_params entry for {pair}; using default k={DEFAULT_PARAMS[0]}, De={DEFAULT_PARAMS[1]})")
    a = math.sqrt(k / (2.0 * De))
    return k, De, a


def morse_energy(r, re, De, a):
    return De * (1.0 - math.exp(-a * (r - re))) ** 2


## 5. Diabatic energy of each state at a given geometry

State 1 uses each bond's equilibrium length from the **RS** geometry; state 2 uses each bond's equilibrium length from the **PS** geometry. Both are then evaluated at whatever geometry you pass in (RS, TS, or PS).

In [5]:
def state_bonds(cfg: StructureSet, kinds: set[str]) -> list[BondChange]:
    return [bc for bc in cfg.bond_changes if bc.kind in kinds]


def diabatic_energy(bonds: list[BondChange], ref_labels, ref_coords, eval_labels, eval_coords) -> float:
    '''Sum of Morse terms for `bonds`, with re taken from the reference
    geometry (RS for state 1, PS for state 2) and r evaluated at `eval_coords`.
    Returns energy in kcal/mol, relative to the reference geometry (0 there).
    '''
    total = 0.0
    for bc in bonds:
        k, De, a = morse_params(ref_labels, bc)
        re = distance(ref_coords, bc.serial_i, bc.serial_j)
        r = distance(eval_coords, bc.serial_i, bc.serial_j)
        total += morse_energy(r, re, De, a)
    return total


def bond_morse_records(step: str, state_label: str, bonds: list[BondChange], ref_labels, ref_coords) -> list[dict]:
    '''One row per bond describing the Morse parameters actually used for
    `state_label` ("state1_RS" or "state2_PS") in this step, so they can be
    inspected/saved/reused later (e.g. when building the full EVB/MM input).
    '''
    records = []
    for bc in bonds:
        ei = ref_labels[bc.serial_i - 1].capitalize()
        ej = ref_labels[bc.serial_j - 1].capitalize()
        k, De, a = morse_params(ref_labels, bc)
        re = distance(ref_coords, bc.serial_i, bc.serial_j)
        records.append({
            "step": step,
            "state": state_label,
            "kind": bc.kind,
            "serial_i": bc.serial_i,
            "serial_j": bc.serial_j,
            "element_i": ei,
            "element_j": ej,
            "note": bc.note,
            "k_kcal_per_mol_A2": k,
            "De_kcal_per_mol": De,
            "a_per_A": a,
            "re_A": re,
        })
    return records


## 6. Fit Delta and H12 per step

In [6]:
def fit_delta_and_h12(step: str, qm_row: pd.Series) -> dict:
    cfg = STRUCTURE_SETS[step]

    geoms = {}
    for role in ROLES:
        path = geometry_path(step, role)
        if not path.exists():
            raise FileNotFoundError(f"Missing optimized geometry for {step} {role}:\n{path}")
        geoms[role] = read_xyz(path)  # (labels, coords)

    rs_labels, rs_coords = geoms["RS"]
    ts_labels, ts_coords = geoms["TS"]
    ps_labels, ps_coords = geoms["PS"]

    bonds1 = state_bonds(cfg, STATE1_KINDS)
    bonds2 = state_bonds(cfg, STATE2_KINDS)

    # Diabatic energies (kcal/mol), each state referenced to 0 at its own geometry.
    e1_rs = diabatic_energy(bonds1, rs_labels, rs_coords, rs_labels, rs_coords)   # = 0 by construction
    e1_ts = diabatic_energy(bonds1, rs_labels, rs_coords, ts_labels, ts_coords)
    e1_ps = diabatic_energy(bonds1, rs_labels, rs_coords, ps_labels, ps_coords)

    e2_rs = diabatic_energy(bonds2, ps_labels, ps_coords, rs_labels, rs_coords)
    e2_ts = diabatic_energy(bonds2, ps_labels, ps_coords, ts_labels, ts_coords)
    e2_ps = diabatic_energy(bonds2, ps_labels, ps_coords, ps_labels, ps_coords)   # = 0 by construction

    # QM adiabatic energies, referenced to RS = 0.
    E_RS = qm_row["E_RS_Eh"] * HARTREE_TO_KCAL
    E_TS = qm_row["E_TS_Eh"] * HARTREE_TO_KCAL
    E_PS = qm_row["E_PS_Eh"] * HARTREE_TO_KCAL

    dE_ts_qm = E_TS - E_RS
    dE_rxn_qm = E_PS - E_RS

    # --- Delta: from the PS asymptote, assuming state 2 dominates there ---
    # Eg(PS) ~= e2(PS) + Delta  =>  Delta = dE_rxn_qm - e2_ps
    delta = dE_rxn_qm - e2_ps

    # --- H12: invert the ground-state formula at the TS ---
    e1 = e1_ts
    e2 = e2_ts + delta
    avg = 0.5 * (e1 + e2)
    half_gap = 0.5 * (e1 - e2)
    discriminant = (avg - dE_ts_qm) ** 2 - half_gap ** 2

    h12_physical = discriminant >= 0
    h12 = math.sqrt(discriminant) if h12_physical else float("nan")

    fit = {
        "step": step,
        "n_state1_bonds": len(bonds1),
        "n_state2_bonds": len(bonds2),
        "E_RS_kcal": E_RS,
        "E_TS_kcal": E_TS,
        "E_PS_kcal": E_PS,
        "dE_ts_qm_kcal": dE_ts_qm,
        "dE_rxn_qm_kcal": dE_rxn_qm,
        "e1_TS_kcal": e1_ts,
        "e2_TS_raw_kcal": e2_ts,
        "e2_PS_raw_kcal": e2_ps,
        "delta_kcal": delta,
        "h12_kcal": h12,
        "h12_physical": h12_physical,
    }

    # Morse parameters actually used for each diabatic state in this step,
    # so they can be saved/inspected/reused (e.g. for the EVB/MM force field).
    morse_records = (
        bond_morse_records(step, "state1_RS", bonds1, rs_labels, rs_coords)
        + bond_morse_records(step, "state2_PS", bonds2, ps_labels, ps_coords)
    )

    return fit, morse_records


## 7. Run the fit for all steps

In [7]:
results = []
all_morse_records = []

for step in STEPS:
    print(f"\n=== {step} ===")

    qm_row = qm_energies_df[qm_energies_df["step"] == step]
    if qm_row.empty or qm_row.isna().any(axis=None):
        print("  Skipped: missing QM energies for this step (RS/TS/PS).")
        continue
    qm_row = qm_row.iloc[0]

    try:
        fit, morse_records = fit_delta_and_h12(step, qm_row)
    except FileNotFoundError as exc:
        print(f"  Skipped: {exc}")
        continue

    if fit["h12_physical"]:
        print(f"  Delta = {fit['delta_kcal']:.2f} kcal/mol")
        print(f"  H12   = {fit['h12_kcal']:.2f} kcal/mol")
    else:
        print("  WARNING: discriminant < 0 -> no real H12 from this simple model.")
        print("  This usually means the TS geometry isn't near the diabatic crossing")
        print("  seam in this reduced bond-only model, or the default Morse")
        print("  parameters are off for this step. Inspect e1_TS / e2_TS_raw / delta.")
        print(f"  Delta = {fit['delta_kcal']:.2f} kcal/mol")

    results.append(fit)
    all_morse_records.extend(morse_records)

evb_fit_df = pd.DataFrame(results)
morse_params_df = pd.DataFrame(all_morse_records)
evb_fit_df



=== RS1_1_to_TS1_2 ===
  This usually means the TS geometry isn't near the diabatic crossing
  seam in this reduced bond-only model, or the default Morse
  parameters are off for this step. Inspect e1_TS / e2_TS_raw / delta.
  Delta = 2.20 kcal/mol

=== TS1_2_to_PS1_2b ===
  Skipped: missing QM energies for this step (RS/TS/PS).

=== RS1_2b_to_PS1_3 ===
  Skipped: missing QM energies for this step (RS/TS/PS).

=== RS2_1_to_TS2_1a ===
  Skipped: missing QM energies for this step (RS/TS/PS).

=== TS2_1a_to_PS2_2 ===
    (no bond_params entry for ('H', 'O'); using default k=400.0, De=80.0)
    (no bond_params entry for ('H', 'O'); using default k=400.0, De=80.0)
    (no bond_params entry for ('H', 'O'); using default k=400.0, De=80.0)
    (no bond_params entry for ('H', 'O'); using default k=400.0, De=80.0)
    (no bond_params entry for ('H', 'O'); using default k=400.0, De=80.0)
    (no bond_params entry for ('H', 'O'); using default k=400.0, De=80.0)
    (no bond_params entry for ('H',

,step,n_state1_bonds,n_state2_bonds,E_RS_kcal,E_TS_kcal,E_PS_kcal,dE_ts_qm_kcal,dE_rxn_qm_kcal,e1_TS_kcal,e2_TS_raw_kcal,e2_PS_raw_kcal,delta_kcal,h12_kcal,h12_physical
0,RS1_1_to_TS1_2,1,2,-52359.886205,-52358.760011,-52357.683367,1.126194,2.202838,0.000097,2.030270,0.0,2.202838,NaN,False
1,TS2_1a_to_PS2_2,2,3,-69193.792094,-69197.578206,-69211.167770,-3.786112,-17.375676,0.597224,93.107778,0.0,-17.375676,18.669629,True


In [8]:
morse_params_df


,step,state,kind,serial_i,serial_j,element_i,element_j,note,k_kcal_per_mol_A2,De_kcal_per_mol,a_per_A,re_A
0,RS1_1_to_TS1_2,state1_RS,weakened,10,16,C,O,ENL.C10=O1 carbonyl weakens,320.0,85.0,1.371989,1.219200
1,RS1_1_to_TS1_2,state2_PS,formed,1,10,N,C,PAF.N2 attacks ENL.C10; N-C bond formation,337.0,73.0,1.519282,2.348680
2,RS1_1_to_TS1_2,state2_PS,weakened,10,16,C,O,ENL.C10=O1 carbonyl weakens,320.0,85.0,1.371989,1.225547
3,TS2_1a_to_PS2_2,state1_RS,broken,20,50,C,H,IND.C3-H50 breaks,340.0,99.0,1.310409,2.550092
4,TS2_1a_to_PS2_2,state1_RS,broken,27,58,O,H,W1 O-H58 breaks,400.0,80.0,1.581139,0.959750
5,TS2_1a_to_PS2_2,state2_PS,formed,60,50,O,H,W2 accepts H50,400.0,80.0,1.581139,0.959083
6,TS2_1a_to_PS2_2,state2_PS,formed,11,58,C,H,ENL.C11 accepts H58,340.0,99.0,1.310409,1.088340
7,TS2_1a_to_PS2_2,state2_PS,double-bond formed,1,10,N,C,N=C/enamine bond order changes,615.0,147.0,1.446318,1.445914


## 8. Save results

In [9]:
FIT_CSV = STAGE2_DIR / "evb_delta_h12_fit.csv"
MORSE_CSV = STAGE2_DIR / "evb_morse_parameters.csv"

evb_fit_df.to_csv(FIT_CSV, index=False)
morse_params_df.to_csv(MORSE_CSV, index=False)

print("Saved:", FIT_CSV)
print("Saved:", MORSE_CSV)


Saved: D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\xtb_preparation\stage2_five_xtb_protocols\evb_delta_h12_fit.csv
Saved: D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\xtb_preparation\stage2_five_xtb_protocols\evb_morse_parameters.csv


## 9. (Optional) Diabatic profile sanity check

Interpolates Cartesian coordinates linearly between RS and PS geometries, evaluates e1(R), e2(R)+Delta, and the EVB ground state Eg(R) along the way, and overlays your 3 actual QM points. This is a quick visual gut-check, not a real reaction-coordinate scan.

In [10]:
import matplotlib.pyplot as plt


def interpolated_geometry(rs_coords, ps_coords, lam: float):
    coords = [
        (
            (1 - lam) * rx + lam * px,
            (1 - lam) * ry + lam * py,
            (1 - lam) * rz + lam * pz,
        )
        for (rx, ry, rz), (px, py, pz) in zip(rs_coords, ps_coords)
    ]
    return coords


def plot_step_profile(step: str, n_points: int = 21):
    cfg = STRUCTURE_SETS[step]
    row = evb_fit_df[evb_fit_df["step"] == step]
    if row.empty:
        print(f"No fit available for {step}; run the fit cell first.")
        return
    row = row.iloc[0]

    rs_labels, rs_coords = read_xyz(geometry_path(step, "RS"))
    ps_labels, ps_coords = read_xyz(geometry_path(step, "PS"))

    bonds1 = state_bonds(cfg, STATE1_KINDS)
    bonds2 = state_bonds(cfg, STATE2_KINDS)

    lambdas = np.linspace(0, 1, n_points)
    e1_curve, e2_curve, eg_curve = [], [], []

    for lam in lambdas:
        coords = interpolated_geometry(rs_coords, ps_coords, lam)
        e1 = diabatic_energy(bonds1, rs_labels, rs_coords, rs_labels, coords)
        e2 = diabatic_energy(bonds2, ps_labels, ps_coords, ps_labels, coords) + row["delta_kcal"]
        h12 = row["h12_kcal"] if row["h12_physical"] else 0.0
        avg, half_gap = 0.5 * (e1 + e2), 0.5 * (e1 - e2)
        eg = avg - math.sqrt(half_gap ** 2 + h12 ** 2)
        e1_curve.append(e1)
        e2_curve.append(e2)
        eg_curve.append(eg)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(lambdas, e1_curve, "--", label="diabatic state 1 (RS-like)")
    ax.plot(lambdas, e2_curve, "--", label="diabatic state 2 (PS-like) + Delta")
    ax.plot(lambdas, eg_curve, "-", linewidth=2, label="EVB ground state")
    ax.scatter([0, 0.5, 1], [row["E_RS_kcal"] - row["E_RS_kcal"], row["dE_ts_qm_kcal"], row["dE_rxn_qm_kcal"]],
               color="black", zorder=5, label="QM RS/TS/PS (illustrative x-position)")
    ax.set_xlabel("linear interpolation coordinate (RS -> PS)")
    ax.set_ylabel("energy relative to RS (kcal/mol)")
    ax.set_title(step)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


# Example:
# plot_step_profile(STEPS[0])


## Notes / next steps

- **Charges**: your RS/PS charge CSVs (`load_charges(step, "RS")` / `load_charges(step, "PS")`)
  are the natural EVB electrostatics for state 1 / state 2 respectively — assign them to
  the same atoms in each diabatic force-field state when you build the full EVB/MM input,
  interpolated by the EVB mixing coefficient during a simulation.
- **Better Morse parameters**: replace `BOND_PARAMS`/`DOUBLE_BOND_PARAMS` with force
  constants extracted from your actual Hessians (e.g. a projected Cartesian-Hessian
  bond-stretch force constant) once you want more than a first-pass parametrization.
- **More than 3 points**: if you generate a few extra constrained-TS geometries at
  different frozen bond distances (using the fixed `optimize_structure()`), you can
  replace the algebraic 2-point/3-point solve in `fit_delta_and_h12` with a least-squares
  fit of Delta and H12 (and check whether H12 is actually roughly constant along the
  coordinate, which the constant-H12 EVB model assumes).
- **Higher-level energies**: if `SP_RESP_Barriers` gives you DFT-level RS/TS/PS energies,
  swap those into `qm_energies_df` — H12 is only as trustworthy as the barrier you feed it.
